# GRPO Notebook 1 — Smoke Test (10 problems)

**Purpose:** Confirm the full GRPO training loop works without crashing before committing GPU time to the full 200-problem training run.

**Model:** Qwen/Qwen2-1.5B-Instruct | **Quant:** INT8 | 

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────────────────────
# trl: GRPOTrainer
# bitsandbytes: INT8 quantization
# peft: LoRA adapters (train on top of frozen quantized weights)
# accelerate: required backend for trl

!pip install -q "transformers>=4.40.0" "trl>=0.8.6" "peft>=0.10.0" "bitsandbytes>=0.43.0" "accelerate>=0.29.0" "datasets>=2.19.0"
print('Dependencies installed')

In [ ]:
# ── CELL 2: Version check + GPU check ─────────────────────────────────────────

import torch, transformers, trl, peft, bitsandbytes as bnb
print(f'torch:         {torch.__version__}')
print(f'transformers:  {transformers.__version__}')
print(f'trl:           {trl.__version__}')
print(f'peft:          {peft.__version__}')
print(f'bitsandbytes:  {bnb.__version__}')
print(f'CUDA:          {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:           {torch.cuda.get_device_name(0)}')
    print(f'VRAM:          {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('No GPU — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── CELL 3: Mount Google Drive ────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/quantization_research/grpo'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# ── CELL 4: All imports ────────────────────────────────────────────────────────

import re, gc, random, time, os
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import GRPOConfig, GRPOTrainer
print('Imports OK')

In [ ]:
# ── CELL 5: Config — all settings in one place ─────────────────────────────────

MODEL_ID   = 'Qwen/Qwen2-1.5B-Instruct'  
SEED       = 42
SMOKE_N    = 10     # training problems for smoke test only
MAX_TOKENS = 512    # max tokens per generated response
GROUP_SIZE = 4      # G: responses generated per prompt by GRPO
EPOCHS     = 1
LR         = 1e-5

random.seed(SEED)
torch.manual_seed(SEED)
print(f'Smoke test: {SMOKE_N} training problems, G={GROUP_SIZE}')

In [ ]:
# ── CELL 6: Load GSM8K train split and sample smoke problems ───────────────────

print('Loading GSM8K train split...')
ds_train = load_dataset('openai/gsm8k', 'main', split='train')
print(f'Train set size: {len(ds_train)}')

random.seed(SEED)
smoke_indices = random.sample(range(len(ds_train)), SMOKE_N)
smoke_problems = [ds_train[i] for i in smoke_indices]

print(f'Sampled {len(smoke_problems)} problems for smoke test')
print(f'\nExample question: {smoke_problems[0]["question"][:150]}...')
print(f'Example answer:   {smoke_problems[0]["answer"][:100]}...')

In [ ]:
# ── CELL 7: Answer extraction helpers ─────────────────────────────────────────

def extract_ground_truth(answer_str):
    """Extract number after #### from GSM8K gold answer."""
    match = re.search(r'####\s*([\d,\.\-]+)', answer_str)
    if match:
        return match.group(1).replace(',', '').strip()
    return None

def extract_prediction(output_str):
    """Extract final answer from model output. Tries 3 patterns in order."""
    # 1. LaTeX boxed answer: \boxed{42}
    m = re.search(r'\\boxed\{([\d,\.\-]+)\}', output_str)
    if m:
        return m.group(1).replace(',', '').strip()
    # 2. GSM8K format: #### 42
    m = re.search(r'####\s*([\d,\.\-]+)', output_str)
    if m:
        return m.group(1).replace(',', '').strip()
    # 3. Last number in output
    nums = re.findall(r'\b([\d,\.]+)\b', output_str)
    if nums:
        return nums[-1].replace(',', '').strip()
    return None

# Quick sanity check
print('Ground truth test:', extract_ground_truth('She had 5. #### 8'))
print('Prediction test:  ', extract_prediction('Therefore \\boxed{42}'))

In [ ]:
# ── CELL 8: Prompt formatting ──────────────────────────────────────────────────

def format_prompt(question):
    return (
        '<|im_start|>system\n'
        'You are a math problem solver. Show your reasoning step by step. '
        'End your answer with the final number in \\boxed{}.\n'
        '<|im_end|>\n'
        f'<|im_start|>user\n{question}<|im_end|>\n'
        '<|im_start|>assistant\n'
    )

print('Prompt example:')
print(format_prompt(smoke_problems[0]['question'])[:300])

In [ ]:
# ── CELL 9: Build HuggingFace Dataset for GRPOTrainer ─────────────────────────

def build_grpo_dataset(problems):
    rows = []
    for p in problems:
        gt = extract_ground_truth(p['answer'])
        if gt is None:
            continue  # skip if answer format is unexpected
        rows.append({
            'prompt': format_prompt(p['question']),
            'ground_truth': gt,
        })
    return Dataset.from_list(rows)

smoke_dataset = build_grpo_dataset(smoke_problems)
print(f'Dataset size: {len(smoke_dataset)}')
print(f'Columns:      {smoke_dataset.column_names}')
print(f'First GT:     {smoke_dataset[0]["ground_truth"]}')

In [ ]:
# ── CELL 10: Reward function ───────────────────────────────────────────────────
# This drives GRPO learning — correct answer = 1.0, wrong = 0.0

def reward_fn(prompts, completions, ground_truth=None, **kwargs):
    rewards = []
    for i, completion in enumerate(completions):
        gt   = ground_truth[i % len(ground_truth)] if ground_truth else None
        pred = extract_prediction(completion)
        correct = (pred is not None and gt is not None
                   and pred.strip() == gt.strip())
        rewards.append(1.0 if correct else 0.0)

    n_correct = int(sum(rewards))
    print(f'  Reward batch: {n_correct}/{len(rewards)} correct '
          f'({100*n_correct/max(len(rewards),1):.0f}%)')
    return rewards

print('Reward function ready')

In [ ]:
# ── CELL 11: Load model with INT8 quantization + LoRA ─────────────────────────


print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

print('Loading model in INT8...')
bnb_cfg = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map='auto',
    trust_remote_code=True,
)
print(f'GPU mem after load: {torch.cuda.memory_allocated()/1e9:.2f} GB')

# Prepare for kbit training (enables gradient flow through quantized layers)
model = prepare_model_for_kbit_training(model)
print('Model ready')

In [ ]:
# ── CELL 12: GRPO training config ─────────────────────────────────────────────
grpo_cfg = GRPOConfig(
    output_dir                  = os.path.join(OUTPUT_DIR, 'smoke_ckpt'),
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_generations             = GROUP_SIZE,
    max_completion_length       = MAX_TOKENS,
    temperature                 = 0.7,
    learning_rate               = LR,
    optim                       = 'adamw_torch',
    gradient_checkpointing      = True,
    fp16                        = False,   # changed
    bf16                        = True,
    logging_steps               = 1,
    save_steps                  = 9999,
    report_to                   = 'none',
)

# LoRA config — passed to trainer directly in TRL 1.2.0
lora_cfg = LoraConfig(
    r                = 8,
    lora_alpha       = 32,
    target_modules   = ['q_proj', 'v_proj'],
    lora_dropout     = 0.05,
    bias             = 'none',
    task_type        = 'CAUSAL_LM',
)

print('Configs ready')

In [ ]:
import trl, inspect
print(trl.__version__)
# Show what GRPOTrainer actually accepts
print(inspect.signature(trl.GRPOTrainer.__init__))

In [ ]:
# ── CELL 13: Run smoke test ────────────────────────────────────────────────────

print('='*55)
print('GRPO SMOKE TEST — 10 problems, 1 epoch')
print('='*55)
t0 = time.time()

trainer = GRPOTrainer(
    model              = model,
    args               = grpo_cfg,
    train_dataset      = smoke_dataset,
    reward_funcs       = reward_fn,
    processing_class   = tokenizer,   # renamed from 'tokenizer' in TRL 1.2.0
    peft_config        = lora_cfg,    # LoRA applied internally by trainer now
)
trainer.train()

elapsed  = (time.time() - t0) / 60
peak_mem = torch.cuda.max_memory_allocated() / 1e9

print('='*55)
print(f'SMOKE TEST COMPLETE')
print(f'  Time:      {elapsed:.1f} min')
print(f'  Peak VRAM: {peak_mem:.2f} GB')
print(f'  Memory OK: {"YES" if peak_mem < 14 else "WARNING — close to limit"}')
print('='*55)
print()
print('Next steps:')
print('  Reward non-zero + no crash → PASSED, run Notebook 2')
print('  All reward = 0.0           → check reward_fn in Cell 10')
print('  CUDA OOM                   → reduce MAX_TOKENS to 256 in Cell 5')

In [ ]:
# ── CELL 14: Cleanup ──────────────────────────────────────────────────────────

del trainer, model
gc.collect()
torch.cuda.empty_cache()
print(f'GPU mem after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB')
print('Session clean. Open Notebook 2 if smoke test passed.')